In [ ]:
import pandas as pd

county = "dallas"
fips ="113"
city = "dfw"

# load all the different csv files 

# google api data 
google = pd.read_csv(f"google_maps_api/filtered_county_farm_data/cleaned_{county}_farm_data.csv",)
print(f"Google: {len(google)} rows, columns: {list(google.columns)}")
google = google.rename(columns={
    "Name" : "name",
    "Address" : "address",
    "Latitude": "lat",
    "Longitude": "lon"
}

# local harvest
local_harvest = pd.read_csv(f"local_harvest_geocoded/{city}_geocoded.csv")
print(f"Local Harvest: {len(local_harvest)} rows, columns: {list(local_harvest.columns)}")
local_harvest = local_harvest.rename(columns={
    "farm_name": "name",
    "latitude": "lat",
    "longitude": "lon",
    "street_address": "address"
})

# open street maps- will need to filter by county later by name 
osm = pd.read_csv("osm_data/urban_farms_with_addresses.csv")
print(f"OSM (full): {len(osm)} rows")
osm = osm.rename(columns={
    "COUNTYFP": "county,
    "latitude": "lat",
    "longitude": "lon"
}

# usda data - will need to sort by county later, by FIPS number 
csa = pd.read_csv("usda_csa.csv")
print(f"csa (full): {len(csa)} rows")
csa.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
}

ofm = pd.read_csv("usda_ofm")
print(f"ofm (full): {len(ofm)} rows")
ofm.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
}

# texas local food data - need to sort by county name 
tx = pd.read_csv("tx_local_food_direct")
print(f"tx (full): {len(tx)} rows")
tx.rename(columns={
    "farm_name": "name",
    "latitude": "lat",
    "longitude": "lon",
    "street_address":"address"
}

# farmers market local data -need to sort by county name
market = pd.read_csv("farmers_market_local_list.csv")
print(f"market (full): {len(market)} rows")
market.rename(columns={
    "farm_name":"name",
    "latitude": "lat",
    "longitude": "lon",
    "street_address":"address"
}
# original dallas data 
dallas = pd.read_csv("cleaned_dallas_geojson.csv")
dallas[["lat", "lon"]] = dallas["coordinates"].str.split(",", expand=True).astype(float)
print(f"dallas : {len(dallas)} rows")
dallas.rename(columns={
    "latitude": "lat",
    "longitude": "lon"
}

In [ ]:
# filtering the csv which contain all counties 
county_osm = osm[osm["county"] == county.capitalize()]
print(f"OSM filtered to '{county.capitalize()}': {len(county_osm)} rows")

county_csa = csa[csa["fips"].astype(str) == fips]
print(f"csa filtered to '{county.capitalize()}': {len(county_csa)} rows")

county_ofm = ofm[ofm["fips"].astype(str) == fips]
print(f"ofm filtered to '{county.capitalize()}': {len(county_ofm)} rows")

county_tx = tx[tx["county"] == county.capitalize()]
print(f"tx filtered to '{county.capitalize()}': {len(county_tx)} rows")

county_market = market[market["county"] == county.capitalize()]
print(f" market filtered to '{county.capitalize()}': {len(county_market)} rows")

In [ ]:
# combine all the data

combined = pd.concat([google, local_harvest, county_osm, county_csa, county_ofm, county_tx, county_market, dallas], ignore_index=True)
print(f"\n Combined (before dedup): {len(combined)} rows")
print(f" Columns: {list(combined.columns)}")

In [ ]:
# --- Drop rows missing lat/lon (can't cluster them) ---
combined = combined.dropna(subset=["lat", "lon"])
print(f"📦 After dropping missing coords: {len(combined)} rows")

# --- 1. Convert lat/lon to meters for DBSCAN ---
lat_rad = np.radians(combined["lat"].mean())
coords_meters = combined[["lat", "lon"]].copy()
coords_meters["lat_m"] = combined["lat"] * 111320
coords_meters["lon_m"] = combined["lon"] * (111320 * np.cos(lat_rad))

# --- 2. DBSCAN to group nearby farms ---
clustering = DBSCAN(eps=500, min_samples=1).fit(coords_meters[["lat_m", "lon_m"]])
combined["cluster_id"] = clustering.labels_
print(f"🗺️  DBSCAN found {combined['cluster_id'].nunique()} geographic clusters")

# --- 3. Clean names for fuzzy matching ---
combined["clean_name"] = (
    combined["name"]
    .astype(str)
    .str.lower()
    .str.replace(r"[^\w\s]", "", regex=True)
    .str.strip()
)

# --- 4. Fuzzy match within each cluster ---
indices_to_drop = set()
fuzzy_threshold = 85

for cluster_id, group in combined.groupby("cluster_id"):
    if len(group) < 2:
        continue
    records = list(group["clean_name"].items())
    print(f"\n🔍 Cluster {cluster_id} ({len(records)} farms): {[n for _, n in records]}")
    for i in range(len(records)):
        idx1, name1 = records[i]
        if idx1 in indices_to_drop:
            continue
        for j in range(i + 1, len(records)):
            idx2, name2 = records[j]
            if idx2 in indices_to_drop:
                continue
            score = fuzz.token_set_ratio(str(name1), str(name2))
            if score >= fuzzy_threshold:
                print(f"   ✂️  DROP '{name2}' (matches '{name1}', score={score})")
                indices_to_drop.add(idx2)

# --- 5. Drop duplicates and clean up ---
deduped = combined.drop(index=list(indices_to_drop)).drop(columns=["cluster_id", "clean_name"])
print(f"\n✅ After fuzzy dedup: {len(deduped)} rows  ({len(combined) - len(deduped)} removed)")

deduped.to_csv(f"deduplicated_{county}_farms.csv", index=False)
print(f"💾 Saved to deduplicated_{county}_farms.csv")
